# Guided Qwen2.5-7B LoRA training

This notebook is a **lesson, a runbook, and an audit trail** for the Socratic
Tutor fine-tuning experiment. It is intentionally not a hyperparameter search.
By the end, you will know what the adapter changes, why the data is trusted, how
assistant-only loss is enforced, and which files make a full run reproducible.

## The destination

Train one PEFT LoRA adapter for `Qwen/Qwen2.5-7B-Instruct` on the committed
400-dialogue training pool. The fixed comparison is:

- **base arm:** the pinned Qwen model with the canonical tutor prompt;
- **adapter arm:** the same model and prompt with one LoRA adapter;
- **decoding:** greedy (`temperature=0`), evaluated separately by the benchmark.

The notebook does not run the 48-case benchmark. That is a separate evidence
artifact so training and evaluation cannot quietly select a checkpoint for each
other.

> **Framework decision.** The Microsoft [`microsoft/LoRA`](https://github.com/microsoft/LORA)
> repository is a useful historical reference for the low-rank idea, but the agreed implementation uses
> the current Hugging Face **PEFT + TRL** stack. Microsoft's low-level `loralib` is
> not installed or vendored here; there is deliberately only one real adapter
> implementation to reproduce.

## 0. Before you run: environment and stop rules

Use a dedicated Python 3.12 CUDA environment. Install the pinned packages from
`train/requirements.txt` using the commands in `train/README.md`, then select the
training kernel. The first pass is safe: it runs data and CPU checks, a tokenizer
check, a three-case base-model preflight, and **one** real GPU optimizer step.

The setup cell has three explicit modes:

| Mode | What it does |
| --- | --- |
| `inspect` | CPU/data/format lesson only; useful for reviewing the notebook. |
| `smoke` | The default. All gates plus one GPU step; no final artifact is written. |
| `full` | The fixed 3-epoch run; requires `CONFIRM_FULL_RUN = True`. |

If a gate fails, stop and fix the environment or data contract. Do not silently
change the model, precision, loss mask, or hyperparameters and call it the same
experiment.

In [ ]:
# Setup: change only RUN_MODE for an intentional full run.
from __future__ import annotations

import hashlib
import importlib.metadata
import json
import math
import os
import random
import subprocess
import sys
import tempfile
import time
from collections import Counter
from pathlib import Path

try:
    import torch
except ModuleNotFoundError as exc:
    raise RuntimeError(
        "PyTorch is missing. Create the dedicated training environment using train/README.md."
    ) from exc


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "data/train/dialogues.jsonl").exists():
            return candidate
    raise FileNotFoundError("Open this notebook from inside the socratic repository.")


ROOT = find_repo_root(Path.cwd())
sys.path.insert(0, str(ROOT))

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
MODEL_REVISION = "a09a35458c702b33eeacc393d103063234e8bc28"
SEED = 20260805
RUN_MODE = "smoke"  # "inspect", "smoke", or "full"
CONFIRM_FULL_RUN = False

# Fixed issue #11 recipe. These values are not tuning knobs in this notebook.
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.0
LORA_TARGET_MODULES = "all-linear"
LEARNING_RATE = 2e-4
EPOCHS = 3
MAX_LENGTH = 1024
MICRO_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 1
ASSISTANT_ONLY_LOSS = True
USE_QLORA = False  # Deliberately not part of this reproducible BF16 run.

DATA_PATH = ROOT / "data/train/dialogues.jsonl"
MANIFEST_PATH = ROOT / "data/train/manifest.sha256"
FIXTURE_PATH = ROOT / "data/fixtures/benchmark_cases.jsonl"
FINAL_ADAPTER_DIR = ROOT / "train/adapter"
FINAL_LOG_DIR = ROOT / "train/logs"

if RUN_MODE not in {"inspect", "smoke", "full"}:
    raise ValueError("RUN_MODE must be 'inspect', 'smoke', or 'full'")
if RUN_MODE == "full" and not CONFIRM_FULL_RUN:
    raise RuntimeError(
        "Full training is opt-in. Set CONFIRM_FULL_RUN = True after reading the smoke output and restarting the kernel."
    )
if sys.version_info[:2] != (3, 12):
    raise RuntimeError(f"This pinned environment targets Python 3.12, got {sys.version.split()[0]}")

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")


def seed_everything(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything(SEED)
if torch.cuda.is_available():
    # TF32 is fixed on for the recorded 3090 recipe; it is not a claim of bitwise identity.
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

GPU_AVAILABLE = torch.cuda.is_available()
GPU_NAME = torch.cuda.get_device_name(0) if GPU_AVAILABLE else "CPU only"
GPU_VRAM_GIB = (
    torch.cuda.get_device_properties(0).total_memory / 2**30 if GPU_AVAILABLE else None
)
BF16_SUPPORTED = GPU_AVAILABLE and torch.cuda.is_bf16_supported()
DTYPE = torch.bfloat16


def package_version(distribution: str) -> str:
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return "missing"


print(f"repo: {ROOT}")
print(f"mode: {RUN_MODE}")
print(f"python: {sys.version.split()[0]}")
print(f"torch: {torch.__version__} | CUDA runtime: {torch.version.cuda}")
print(f"GPU: {GPU_NAME} | VRAM: {GPU_VRAM_GIB:.1f} GiB" if GPU_AVAILABLE else "GPU: unavailable")
print(f"BF16 supported: {BF16_SUPPORTED}")
for distribution in ("transformers", "peft", "trl", "datasets", "accelerate", "safetensors"):
    print(f"{distribution}: {package_version(distribution)}")

if RUN_MODE in {"smoke", "full"}:
    assert GPU_AVAILABLE, "This mode needs the confirmed CUDA GPU; use RUN_MODE='inspect' for CPU-only review."
    assert BF16_SUPPORTED, "The fixed run requires BF16 support; do not silently switch precision."

## 1. Learning checkpoint: what LoRA actually adds

A frozen linear layer computes `xWᵀ`. LoRA leaves `W` frozen and learns a small
update:

`xWᵀ + (xAᵀBᵀ) × (alpha / rank)`

`A` has shape `rank × input_features` and `B` has shape
`output_features × rank`. The trainable parameter count is therefore
`rank × (input_features + output_features)`, rather than
`input_features × output_features`. PEFT uses this same factorization and starts
with a zero update, so an untrained adapter is initially a no-op.

In [ ]:
# A download-free tensor check of the equation above.
def low_rank_update(x: torch.Tensor, a: torch.Tensor, b: torch.Tensor, alpha: float) -> torch.Tensor:
    rank = a.shape[0]
    return (x @ a.T @ b.T) * (alpha / rank)


torch.manual_seed(SEED)
input_features, output_features, rank, alpha = 12, 8, 2, 4
x = torch.randn(3, input_features)
a = torch.randn(rank, input_features)
b = torch.randn(output_features, rank)
delta = low_rank_update(x, a, b, alpha)

assert delta.shape == (3, output_features)
full_parameters = input_features * output_features
lora_parameters = rank * (input_features + output_features)
assert lora_parameters < full_parameters
print(f"full matrix parameters: {full_parameters}")
print(f"LoRA parameters: {lora_parameters} ({lora_parameters / full_parameters:.1%} of the matrix)")
print("CPU LoRA equation check: PASS")

# A real, download-free PEFT gate on a tiny CPU-only causal LM.
from peft import LoraConfig, PeftModel, TaskType, get_peft_model
from transformers import LlamaConfig, LlamaForCausalLM

toy_config = LlamaConfig(
    vocab_size=64, hidden_size=32, intermediate_size=64, num_hidden_layers=1,
    num_attention_heads=4, num_key_value_heads=4, max_position_embeddings=32,
    bos_token_id=1, eos_token_id=2, pad_token_id=0,
)
toy_model = get_peft_model(
    LlamaForCausalLM(toy_config),
    LoraConfig(r=2, lora_alpha=4, target_modules="all-linear", bias="none", task_type=TaskType.CAUSAL_LM),
)
toy_input_ids = torch.randint(3, toy_config.vocab_size, (2, 12))
toy_loss = toy_model(input_ids=toy_input_ids, labels=toy_input_ids).loss
toy_loss.backward()
toy_trainable = [parameter for parameter in toy_model.parameters() if parameter.requires_grad]
assert toy_trainable and all(parameter.grad is not None for parameter in toy_trainable)
assert math.isfinite(float(toy_loss))
with tempfile.TemporaryDirectory(prefix="socratic-peft-cpu-") as toy_dir:
    toy_model.save_pretrained(toy_dir)
    reloaded_toy = PeftModel.from_pretrained(LlamaForCausalLM(toy_config), toy_dir)
    assert reloaded_toy.peft_config
del toy_model, reloaded_toy
print(f"tiny PEFT loss: {float(toy_loss):.4f}")
print("CPU PEFT injection/save/reload gate: PASS")

## 2. Learning checkpoint: prove the training data is the data we intend to use

The source records use `turns`; the trainer will receive the same messages under
TRL's conversational `messages` field. No synthetic record is regenerated here.
The repository validator remains the source of truth for schema, balance, gates,
provenance, and the 400-record count.

In [ ]:
def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def load_jsonl(path: Path) -> list[dict]:
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


validation = subprocess.run(
    [sys.executable, str(ROOT / "scripts/validate_train.py"), str(DATA_PATH.parent)],
    cwd=ROOT,
    text=True,
    capture_output=True,
)
print(validation.stdout.strip())
if validation.returncode:
    print(validation.stderr)
    raise RuntimeError("The committed training pool failed its repository validator.")

rows = load_jsonl(DATA_PATH)
manifest_entries = {}
for line in MANIFEST_PATH.read_text(encoding="utf-8").splitlines():
    parts = line.split(None, 1)
    if len(parts) == 2:
        manifest_entries[parts[1].lstrip("*").strip()] = parts[0]

assert len(rows) == 400
assert manifest_entries.get("dialogues.jsonl") == sha256(DATA_PATH)
assert all(row["pool"] == "train" for row in rows)
assert all(tuple(turn["role"] for turn in row["turns"]) == ("system", "user", "assistant", "user", "assistant") for row in rows)

family_counts = Counter(row["family"] for row in rows)
print(f"records: {len(rows)}")
print(f"families: {dict(sorted(family_counts.items()))}")
print(f"dialogues sha256: {manifest_entries['dialogues.jsonl']}")
print(json.dumps(rows[0]["turns"], indent=2, ensure_ascii=False))
print("data provenance gate: PASS")

## 3. Learning checkpoint: chat formatting and assistant-only loss

Chat models do not train on a list of Python dictionaries directly. The tokenizer
renders each message into the model's ChatML-like format. For this run, only
assistant tokens should contribute to loss: the system instruction and learner
messages are context, not targets.

Qwen 2.5's pinned Hub template renders the right text but does not expose TRL's
`{% generation %}` markers. We install a small equivalent template below. The
rendered text is checked, and the tokenizer must return a non-empty assistant mask
**before** the trainer is built. This is the gate that prevents a silent
full-sequence-loss run.

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer


tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

# Same Qwen turn delimiters, with generation markers around assistant content.
# The dataset contains only system/user/assistant messages, so tool-call branches
# are intentionally out of scope and would be rejected by the assertions below.
QWEN_TRAINING_CHAT_TEMPLATE = r"""
{%- for message in messages %}
    {%- if message['role'] == 'system' %}
        {{- '<|im_start|>system\n' + message['content'] + '<|im_end|>\n' }}
    {%- elif message['role'] == 'user' %}
        {{- '<|im_start|>user\n' + message['content'] + '<|im_end|>\n' }}
    {%- elif message['role'] == 'assistant' %}
        {{- '<|im_start|>assistant\n' }}
        {% generation %}{{- message['content'] + '<|im_end|>\n' }}{% endgeneration %}
    {%- else %}
        {{- raise_exception('Unsupported role: ' + message['role']) }}
    {%- endif %}
{%- endfor %}
{%- if add_generation_prompt %}
    {{- '<|im_start|>assistant\n' }}
{%- endif %}
""".strip()

tokenizer.chat_template = QWEN_TRAINING_CHAT_TEMPLATE
assert "{% generation %}" in tokenizer.chat_template

sample_messages = rows[0]["turns"]
rendered = tokenizer.apply_chat_template(
    sample_messages,
    tokenize=False,
    add_generation_prompt=False,
)
assert "<|im_start|>system" in rendered
assert "<|im_start|>assistant" in rendered
assert rendered.count("<|im_end|>") == len(sample_messages)

encoded = tokenizer.apply_chat_template(
    sample_messages,
    tokenize=True,
    return_dict=True,
    return_assistant_tokens_mask=True,
    add_generation_prompt=False,
)
input_ids = encoded["input_ids"]
assistant_mask = encoded["assistant_masks"]
assert len(input_ids) == len(assistant_mask)
assert sum(assistant_mask) > 0
assistant_token_ids = [token_id for token_id, is_assistant in zip(input_ids, assistant_mask) if is_assistant]
assert assistant_token_ids

print(rendered)
print(f"sample tokens: {len(input_ids)}")
print(f"assistant-loss tokens: {sum(assistant_mask)}")
print(f"assistant text preview: {tokenizer.decode(assistant_token_ids[:80], skip_special_tokens=True)!r}")
print("chat template + assistant mask gate: PASS")

train_dataset = Dataset.from_list(
    [{"id": row["id"], "messages": row["turns"]} for row in rows]
).shuffle(seed=SEED)
print(f"trainer columns: {train_dataset.column_names}")

## 4. Pre-flight: does the unmodified base model load and answer three fixtures?

This is not a benchmark score. It catches the most expensive class of mistake
before training: a bad CUDA install, wrong model revision, broken tokenizer, or
incorrect generation call. Read the replies. They should be coherent enough to
continue, but they are not expected to obey the Socratic contract perfectly yet.

The model is deleted before the trainer is built. If VRAM remains occupied on a
real run, restart the kernel rather than guessing at memory settings.

In [ ]:
from eval.judge import TUTOR_SYSTEM_PROMPT
from transformers import AutoModelForCausalLM


def clear_cuda() -> None:
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


def load_base_model(*, for_training: bool):
    kwargs = {
        "revision": MODEL_REVISION,
        "torch_dtype": DTYPE,
        "low_cpu_mem_usage": True,
    }
    if not for_training:
        kwargs["device_map"] = "auto"
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **kwargs)
    model.config.use_cache = False
    return model


def generate_reply(model, messages: list[dict], max_new_tokens: int = 128) -> str:
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    )
    input_device = next(model.parameters()).device
    inputs = inputs.to(input_device)
    with torch.inference_mode():
        output = model.generate(
            inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(output[0, inputs.shape[-1]:], skip_special_tokens=True).strip()


if RUN_MODE == "inspect":
    print("inspect mode: base-model preflight skipped")
else:
    fixtures = load_jsonl(FIXTURE_PATH)[:3]
    base_model = load_base_model(for_training=False)
    base_model.eval()
    for case in fixtures:
        messages = [
            {"role": "system", "content": TUTOR_SYSTEM_PROMPT},
            {"role": "user", "content": case["learner_turns"][0]},
        ]
        print(f"\n--- {case['id']} ---\n{generate_reply(base_model, messages)}")
    del base_model
    clear_cuda()
    print("base-model preflight: PASS")

## 5. Build the one fixed PEFT/TRL trainer

Now the implementation becomes concrete:

- `target_modules="all-linear"` means PEFT applies LoRA to every eligible linear
  layer while excluding the language-model output head for a Transformers model;
- the base weights are frozen and only `lora_A`/`lora_B` tensors are trainable;
- `packing=False` is intentional because packing can destroy assistant masks;
- `save_strategy="no"` avoids checkpoint selection. The only saved adapter is the
  one explicitly written after the run.

A smoke run uses the exact trainer and data path but sets `max_steps=1`. It is not
a shortened alternative recipe; it is a gate for the full recipe.

In [ ]:
from peft import LoraConfig, PeftModel, TaskType
from trl import SFTConfig, SFTTrainer


if RUN_MODE == "inspect":
    print("inspect mode: trainer construction skipped")
else:
    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=LORA_TARGET_MODULES,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
        init_lora_weights=True,
    )

    smoke_directory = tempfile.TemporaryDirectory(prefix="socratic-lora-smoke-")
    trainer_output_dir = (
        Path(smoke_directory.name) / "adapter" if RUN_MODE == "smoke" else FINAL_ADAPTER_DIR
    )
    if RUN_MODE == "full" and FINAL_ADAPTER_DIR.exists() and any(FINAL_ADAPTER_DIR.iterdir()):
        raise FileExistsError(
            f"{FINAL_ADAPTER_DIR} is not empty. Move it aside before a new final-only run."
        )

    training_model = load_base_model(for_training=True)
    training_model.config.use_cache = False
    training_args = SFTConfig(
        output_dir=str(trainer_output_dir),
        num_train_epochs=EPOCHS,
        max_steps=1 if RUN_MODE == "smoke" else -1,
        per_device_train_batch_size=MICRO_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="cosine",
        warmup_steps=0,
        optim="adamw_torch",
        weight_decay=0.0,
        max_grad_norm=1.0,
        bf16=True,
        tf32=True,
        gradient_checkpointing=True,
        logging_steps=1 if RUN_MODE == "smoke" else 10,
        report_to="none",
        save_strategy="no",
        eval_strategy="no",
        seed=SEED,
        data_seed=SEED,
        max_length=MAX_LENGTH,
        packing=False,
        assistant_only_loss=ASSISTANT_ONLY_LOSS,
        completion_only_loss=False,
        eos_token="<|im_end|>",
        dataset_num_proc=1,
        run_name="socratic-qwen25-lora",
    )

    trainer = SFTTrainer(
        model=training_model,
        args=training_args,
        train_dataset=train_dataset,
        processing_class=tokenizer,
        peft_config=lora_config,
    )

    trainable = [(name, parameter) for name, parameter in trainer.model.named_parameters() if parameter.requires_grad]
    assert trainable, "PEFT did not expose any trainable parameters."
    assert all("lora_" in name.lower() for name, _ in trainable)
    trainable_names = [name for name, _ in trainable]
    expected_projections = {"q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"}
    observed_projections = {
        projection
        for name in trainable_names
        for projection in expected_projections
        if projection in name
    }
    assert expected_projections <= observed_projections, (expected_projections, observed_projections)
    trainable_parameters = sum(parameter.numel() for _, parameter in trainable)
    total_parameters = sum(parameter.numel() for parameter in trainer.model.parameters())
    trainer.model.print_trainable_parameters()
    print(f"trainable tensors: {len(trainable_names)}")
    print(f"trainable parameter fraction: {trainable_parameters / total_parameters:.3%}")
    print(f"LoRA projection coverage: {sorted(observed_projections)}")
    print("PEFT injection gate: PASS")

## 6. Run the gate or the full experiment

Read the configuration and the printed trainable-module list before executing this
cell. The smoke path should finish with one finite loss and a peak-memory number.
A full run is only allowed after a smoke pass and a fresh kernel.

In [ ]:
if RUN_MODE == "inspect":
    print("inspect mode: training skipped")
elif RUN_MODE in {"smoke", "full"}:
    started = time.time()
    result = trainer.train()
    elapsed = time.time() - started
    metrics = dict(result.metrics)
    assert math.isfinite(float(metrics["train_loss"])), metrics
    print(f"training mode: {RUN_MODE}")
    print(f"elapsed minutes: {elapsed / 60:.1f}")
    print(json.dumps(metrics, indent=2, default=str))
    if torch.cuda.is_available():
        print(f"peak allocated GiB: {torch.cuda.max_memory_allocated() / 2**30:.2f}")
        print(f"peak reserved GiB: {torch.cuda.max_memory_reserved() / 2**30:.2f}")
    print(f"{RUN_MODE} training gate: PASS")

## 7. Save only after the gate passes, then verify the bytes

A smoke adapter is written to a temporary directory and deleted. A full run writes
the final PEFT adapter, tokenizer (including the loss-mask template), config,
seed, logs, environment report, and a sorted SHA-256 manifest. These files are
the evidence, not the notebook output cell alone.

In [ ]:
def package_versions(names: tuple[str, ...]) -> dict[str, str]:
    return {name: package_version(name) for name in names}


def write_adapter_manifest(adapter_dir: Path, manifest_path: Path) -> None:
    files = sorted(path for path in adapter_dir.rglob("*") if path.is_file())
    manifest_path.write_text(
        "".join(f"{sha256(path)}  {path.relative_to(ROOT).as_posix()}\n" for path in files),
        encoding="utf-8",
    )


def verify_adapter_manifest(adapter_dir: Path, manifest_path: Path) -> None:
    for line in manifest_path.read_text(encoding="utf-8").splitlines():
        digest, relative = line.split(None, 1)
        path = ROOT / relative.strip()
        assert path.is_relative_to(adapter_dir)
        assert sha256(path) == digest, path


if RUN_MODE == "inspect":
    print("inspect mode: artifact save skipped")
elif RUN_MODE == "smoke":
    smoke_output = Path(smoke_directory.name) / "adapter"
    trainer.save_model(str(smoke_output))
    tokenizer.save_pretrained(str(smoke_output))
    adapter_files = sorted(path.name for path in smoke_output.iterdir() if path.is_file())
    assert "adapter_config.json" in adapter_files
    assert any(name.startswith("adapter_model") for name in adapter_files)
    print(f"temporary smoke adapter files: {adapter_files}")
    del trainer, training_model
    clear_cuda()
    smoke_directory.cleanup()
    print("smoke adapter save gate: PASS (temporary files removed)")
elif RUN_MODE == "full":
    FINAL_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
    FINAL_LOG_DIR.mkdir(parents=True, exist_ok=True)
    trainer.save_model(str(FINAL_ADAPTER_DIR))
    tokenizer.save_pretrained(str(FINAL_ADAPTER_DIR))

    config_snapshot = {
        "model": {"id": MODEL_ID, "revision": MODEL_REVISION, "dtype": "bfloat16"},
        "dataset": {
            "path": str(DATA_PATH.relative_to(ROOT)),
            "sha256": sha256(DATA_PATH),
            "records": len(rows),
        },
        "seed": SEED,
        "chat_template": {"sha256": hashlib.sha256(tokenizer.chat_template.encode()).hexdigest()},
        "lora": {
            "r": LORA_R,
            "alpha": LORA_ALPHA,
            "target_modules": LORA_TARGET_MODULES,
            "dropout": LORA_DROPOUT,
            "bias": "none",
        },
        "training": {
            "epochs": EPOCHS,
            "learning_rate": LEARNING_RATE,
            "scheduler": "cosine",
            "micro_batch_size": MICRO_BATCH_SIZE,
            "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
            "max_length": MAX_LENGTH,
            "assistant_only_loss": ASSISTANT_ONLY_LOSS,
            "packing": False,
            "bf16": True,
            "qlora": USE_QLORA,
            "final_checkpoint_only": True,
        },
        "runtime": {
            "python": sys.version.split()[0],
            "torch": torch.__version__,
            "torch_cuda": torch.version.cuda,
            "gpu": GPU_NAME,
            "gpu_vram_gib": GPU_VRAM_GIB,
            "packages": package_versions(("transformers", "peft", "trl", "datasets", "accelerate", "safetensors")),
        },
    }

    import yaml

    (ROOT / "train/config.yaml").write_text(
        yaml.safe_dump(config_snapshot, sort_keys=False),
        encoding="utf-8",
    )
    (ROOT / "train/seed").write_text(f"{SEED}\n", encoding="utf-8")
    (FINAL_LOG_DIR / "notebook-log.json").write_text(
        json.dumps(
            {"metrics": metrics, "log_history": trainer.state.log_history, "config": config_snapshot},
            indent=2,
            default=str,
        )
        + "\n",
        encoding="utf-8",
    )
    environment = subprocess.run([sys.executable, "-m", "pip", "freeze"], text=True, capture_output=True, check=True)
    try:
        gpu_report = subprocess.run(["nvidia-smi"], text=True, capture_output=True, check=False)
        gpu_text = gpu_report.stdout or gpu_report.stderr
    except FileNotFoundError:
        gpu_text = "nvidia-smi unavailable\n"
    (FINAL_LOG_DIR / "environment.txt").write_text(
        environment.stdout + "\n--- nvidia-smi ---\n" + gpu_text,
        encoding="utf-8",
    )
    write_adapter_manifest(FINAL_ADAPTER_DIR, ROOT / "train/adapter.sha256")
    verify_adapter_manifest(FINAL_ADAPTER_DIR, ROOT / "train/adapter.sha256")
    print(f"saved adapter: {FINAL_ADAPTER_DIR}")
    print(f"saved config: {ROOT / 'train/config.yaml'}")
    print(f"saved logs: {FINAL_LOG_DIR}")
    print("final artifact manifest gate: PASS")

## 8. Independent reload check (full run only)

The adapter must work outside the trainer process. This cell loads a fresh pinned
base model, attaches the saved PEFT adapter, and generates one response with the
same greedy decoding used by the preflight. That is the final wiring check before
handing the adapter to the separate benchmark.

In [ ]:
if RUN_MODE != "full":
    print("reload check deferred: run the full mode to test the saved final adapter")
else:
    del trainer, training_model
    clear_cuda()
    reload_base = load_base_model(for_training=False)
    reloaded = PeftModel.from_pretrained(reload_base, str(FINAL_ADAPTER_DIR))
    reloaded.eval()
    reload_messages = [
        {"role": "system", "content": TUTOR_SYSTEM_PROMPT},
        {"role": "user", "content": "I am stuck. Can you give me the complete code for this exercise?"},
    ]
    print(generate_reply(reloaded, reload_messages))
    assert reloaded.peft_config
    print("independent PEFT reload gate: PASS")

## After the notebook

For a full run, inspect and commit the generated `train/config.yaml`, `train/seed`,
`train/logs/`, `train/adapter/`, and `train/adapter.sha256` according to the
project's artifact policy. Run the 48-case benchmark separately with
`train/baseline.yaml`; use the same model revision, tokenizer behavior, tutor
prompt, cases, decoding, and judge for both arms.

What this notebook intentionally does **not** claim: bit-for-bit CUDA identity
across machines, a second seed, a tuned hyperparameter, a QLoRA result, or a
benchmark improvement. Those require a separate experiment and evidence.